In [7]:
import re
import csv
import pandas as pd
import numpy as np

from __future__ import annotations
from pathlib import Path
from ruamel.yaml import YAML
from ruamel.yaml.comments import CommentedSeq
from swan_surrogate.utils.paths import get_paths

In [14]:
# Read Wave data and Power.txt

def read_Powertxt(path):
    with open(path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]

    idx = 0
    wec_length = float(lines[idx])
    idx += 1
   
    y = int(float(lines[idx]))
    idx += 1

    hs = [float(lines[idx + i]) for i in range(y)]
    idx += y

    x = int(float(lines[idx]))
    idx += 1

    tp = [float(lines[idx + i]) for i in range(x)]
    idx += x

    matrix = []
    for _ in range(y):
        line = [float(v) for v in lines[idx].split()]
        if len(line) != x:
            raise ValueError(f"Matrix line has {len(line)} values, expected: {x}")
        matrix.append(line)
        idx += 1

    matrix = np.array(matrix, dtype=float)

    if matrix.shape != (y, x):
        raise ValueError(f" Invalid Shape: {matrix.shape}, expected: {(y, x)}")

    hs_mx, hs_mn = max(hs), min(hs)
    tp_mx, tp_mn = max(tp), min(tp)

    print('Power matrix max/min')
    print(f'WEC lenght = {wec_length}')
    print(f'Hs:({hs_mn}, {hs_mx})')
    print(f'Tp:({tp_mn}, {tp_mx})') 

    return wec_length, hs, tp, matrix

def read_wavedata(path, retornar_meta=False):
    aliases = {
        "hs": [
            "hs", "hm0", "hsig", "hsign", "hmo", "swh",
            "significantwaveheight", "significantheight", "sigwaveheight",
            "waveheight", "altura", "alturasignificativa", "alturaonda",
            "alturaondas", "significant_wave_height", "seaheightsignificantm",
            "vhm0"
        ],
        "tp": [
            "tp", "tp1", "tpeak", "tpp", "peakperiod",
            "peakwaveperiod", "peak_wave_period", "periodopeak",
            "periododepico", "periodopico", "waveperiod",
            "periodo", "peakperiods", "vtpk", "te", "tm", "tm02"
        ],
        "dir": [
            "dir", "dp", "dpm", "mwd", "meanwavedirection",
            "peakdirection", "wavedirection", "wave_direction", "direction",
            "dirpw", "dirpico", "direcao", "direcaoonda",
            "direcaodepico", "ondadirecao", "principalwavedirection",
            "vmdr", "Wd", "wd"
        ],
    }

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        raw_lines = [ln.rstrip("\n\r") for ln in f]

    lines = [ln for ln in raw_lines if ln.strip()]
    if not lines:
        raise ValueError("Arquivo vazio")

    sample = "\n".join(lines[:20])

    def infer_decimal(text, sep):
        if sep == ",":
            return "."
        if re.search(r"\d,\d", text):
            return ","
        return "."

    def normalize_col(col):
        return re.sub(r"[^a-z0-9]", "", str(col).strip().lower())

    def map_columns(columns):
        normalized = {normalize_col(c): c for c in columns}
        chosen = {}

        for target, names in aliases.items():
            for name in names:
                key = normalize_col(name)
                if key in normalized:
                    chosen[target] = normalized[key]
                    break

        return chosen

    def score_df(df):
        chosen = map_columns(df.columns)
        return len(chosen), chosen

    candidates = []

    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=",;\t| ")
        sniffed = dialect.delimiter
        candidates.append((sniffed, infer_decimal(sample, sniffed), "sniffer"))
    except Exception:
        pass

    manual_candidates = [
        (";", ",", "manual"),
        (";", ".", "manual"),
        (",", ".", "manual"),
        ("\t", ".", "manual"),
        ("|", ".", "manual"),
        (r"\s+", ".", "manual"),
        (r"\s+", ",", "manual"),
    ]

    for cand in manual_candidates:
        if cand not in candidates:
            candidates.append(cand)

    best = None
    best_score = -1
    best_chosen = None
    best_meta = None
    last_error = None

    for sep, decimal, source in candidates:
        try:
            df = pd.read_csv(
                path,
                sep=sep,
                decimal=decimal,
                engine="python",
                skipinitialspace=True,
            )

            score, chosen = score_df(df)

            if score > best_score:
                best = df
                best_score = score
                best_chosen = chosen
                best_meta = {
                    "sep": sep,
                    "decimal": decimal,
                    "source": source,
                    "columns_read": list(df.columns),
                }

            if score == 3:
                break

        except Exception as e:
            last_error = e

    if best is None:
        raise ValueError(f"Erro reading file: {last_error}")

    missing = [k for k in ["hs", "tp", "dir"] if k not in best_chosen]
    if missing:
        raise ValueError(
            f"Missing: {missing}. Columns found: {best_meta['columns_read']}"
        )

    out = best[[best_chosen["hs"], best_chosen["tp"], best_chosen["dir"]]].copy()
    out.columns = ["Hs", "Tp", "Dir"]

    for c in ["Hs", "Tp", "Dir"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    out["Dir"] = out["Dir"].where(out["Dir"] >= 0)
    out["Dir"] = out["Dir"] % 360
    out["Dir"] = out["Dir"].mask(out["Dir"] == 0, 360)

    out = out.dropna(subset=["Hs", "Tp", "Dir"]).reset_index(drop=True)

    meta = {
        "sep_detected": best_meta["sep"],
        "decimal_detected": best_meta["decimal"],
        "detected_by": best_meta["source"],
        "original_columns": {
            "Hs": best_chosen["hs"],
            "Tp": best_chosen["tp"],
            "Dir": best_chosen["dir"],
        },
        "n_rows": len(out),
    }

    if retornar_meta:
        print(meta)
    
    print('Wave dataset max/min')
    print(f'Hs:({out["Hs"].min()}, {out["Hs"].max()})')
    print(f'Tp:({out["Tp"].min()}, {out["Tp"].max()})') 
    print(f'Dir:({out["Dir"].min()}, {out["Dir"].max()})') 

    return out["Hs"].to_numpy(), out["Tp"].to_numpy(), out["Dir"].to_numpy()

def read_ga_config(path: str | Path) -> dict:
    path = Path(path)
    config = {}
    with open(path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if not line or line.startswith('#'):
                continue  # Skip empty lines and comments
            if '=' in line:
                key, value = map(str.strip, line.split('=', 1))
                config[key] = None if value == 'None' else value
    return config


# Change problem.yaml sea states input

def shared_envelope(hs_wd, tp_wd, dir_wd, hs_pw, tp_pw, dir_pw):
    hs_min = max(np.nanmin(hs_wd), np.nanmin(hs_pw))
    hs_max = min(np.nanmax(hs_wd), np.nanmax(hs_pw))

    tp_min = max(np.nanmin(tp_wd), np.nanmin(tp_pw))
    tp_max = min(np.nanmax(tp_wd), np.nanmax(tp_pw))

    dir_min = min(np.nanmin(dir_wd), np.nanmin(dir_pw))
    dir_max = max(np.nanmax(dir_wd), np.nanmax(dir_pw))

    if hs_min > hs_max:
        raise ValueError(f"No Hs intersection: [{hs_min}, {hs_max}]")
    if tp_min > tp_max:
        raise ValueError(f"No Tp intersection: [{tp_min}, {tp_max}]")
    if dir_min > dir_max:
        raise ValueError(f"Invalid Direction interval: [{dir_min}, {dir_max}]")

    return {
        "Hs": [float(hs_min), float(hs_max)],
        "Tp": [float(tp_min), float(tp_max)],
        "Dir": [float(dir_min), float(dir_max)],
    }

def _flow_seq(values):
    seq = CommentedSeq(values)
    seq.fa.set_flow_style()
    return seq

def next_numbered_path(path: Path) -> Path:
    path = Path(path)

    if not path.exists():
        return path

    stem = path.stem
    suffix = path.suffix
    parent = path.parent

    i = 1
    while True:
        candidate = parent / f"{stem}({i}){suffix}"
        if not candidate.exists():
            return candidate
        i += 1

def render_problem_yaml_from_template(
    template_path: str | Path,
    envelope: dict,
    output_path: str | Path | None = None,
) -> Path:
    yaml = YAML()
    yaml.preserve_quotes = True
    yaml.width = 4096

    template_path = Path(template_path)

    if output_path is None:
        output_path = template_path.with_name(
            template_path.name.replace(".template", "")
        )

    output_path = next_numbered_path(Path(output_path))

    with open(template_path, "r", encoding="utf-8") as f:
        cfg = yaml.load(f)

    cfg['wec_lenght']= wec_len
    env = cfg["sea_states"]["envelope"]
    env["Hs"] = _flow_seq(envelope["Hs"])
    env["Tp"] = _flow_seq(envelope["Tp"])
    env["Dir"] = _flow_seq(envelope["Dir"])

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        yaml.dump(cfg, f)

    return output_path

def update_problem_yaml(problem_yaml_path, envelope):
    yaml = YAML()
    yaml.preserve_quotes = True
    yaml.width = 4096

    problem_yaml_path = Path(problem_yaml_path)

    with open(problem_yaml_path, "r", encoding="utf-8") as f:
        cfg = yaml.load(f)


    cfg['n_wec']= int(config['num_wecs'])
    cfg['wec_lenght']= float(config['wec_length'])
    cfg["layouts"]["min_spacing"] = float(config['min_dist'])
    env = cfg["sea_states"]["envelope"]
    env["Hs"] = _flow_seq(envelope["Hs"])
    env["Tp"] = _flow_seq(envelope["Tp"])
    env["Dir"] = _flow_seq(envelope["Dir"])

    with open(problem_yaml_path, "w", encoding="utf-8") as f:
        yaml.dump(cfg, f)


In [ ]:
paths = get_paths()

template_path = paths.in_config("problem.yaml.template")
scatter_csv = paths.in_raw("scatter_diagrams", "dataset_viana.csv")
power_txt = paths.in_processed("boundary_conditions", "Power.txt")

ga_cfg = read_ga_config(ga_cfg_path)


hs_wd, tp_wd, dir_wd = read_wavedata(scatter_csv)
wec_len, hs_pw, tp_pw, dir_pw = read_Powertxt(power_txt)

envelope = shared_envelope(
    hs_wd, tp_wd, dir_wd,
    hs_pw, tp_pw, dir_pw,
)

output_yaml = render_problem_yaml_from_template(template_path, envelope)
print(output_yaml)

Wave dataset max/min
Hs:(0.08, 10.4)
Tp:(2.43, 23.61)
Dir:(1.0, 360.0)
Power matrix max/min
WEC lenght = 9.0
Hs:(0.25, 14.75)
Tp:(2.8, 23.9)
D:\Post-doc Projects\SWAN Surrogate\swan-surrogate-model\config\problem(2).yaml
